# JRA-3Q 海面更正気圧（気圧配置・日本域）一括ダウンロード（Colab版）

手元のPC/ネットワークから `github.com` や GDEX（データ配布元）へのHTTPS通信がブロックされる環境向けに、Google Colab上でダウンロードするノートブックです。Colab（Googleのサーバー）から直接ダウンロードするので、手元の回線・セキュリティソフトの制限は関係なくなります。

## 日本域だけに切り出し（OPeNDAP方式）
GDEXのTHREDDSサーバーに **OPeNDAP** で接続し、月ごとのファイルから日本周辺（既定: 北緯15〜50度、東経115〜155度）の範囲だけを読み取って保存します。1ヶ月あたり約4.8MB（全球なら84MB）、全185ヶ月で合計1GB弱です。

## 速度より確実さを優先しています（所要1時間程度）
GDEXのサーバーは負荷に弱く、少しでも重いリクエストを送ると「504 Gateway Time-out」を返してきます。そのため:

- **1件ずつ順番に**処理します（並列にすると大半が504で失敗）
- 1ヶ月分（124時刻）を一度に要求せず、**5日ずつに分割**して取得します（1回のリクエストを軽くすることで、サーバーが時間内に応答できるようにするため）
- 各リクエストは**最大5回まで、待ち時間を倍々にしながら再試行**します
- さらに月単位でも最大3周まで再試行します

それでも504が多発する場合は、③のセルに `--chunk-days 2` を追加してさらに細かく分割してください。

## 保存先について
Googleドライブは使わず、Colab上の一時ディスクにまとめてダウンロードしてから、**最後に1回だけZIPにしてブラウザ経由でPCにダウンロード**します（ブラウザの通信は制限されていないので問題なく動くはずです）。

## セッションが切れたら
スクリプトは既にあるファイルをスキップするので、同じセッション内なら③を再実行すれば続きから再開されます。セッションごと切れた場合は①からやり直してください。

## 使い方
**⑤を先に実行**してから、①→②→③→④の順で実行してください（③に1時間ほどかかるため、先に切断防止を仕込んでおくのがおすすめです）。

## ① リポジトリを取得（初回はクローン、2回目以降は最新化のみ）

In [ ]:
import os

REPO_DIR = '/content/typhoon-wind-rainfall'
BRANCH = 'claude/typhoon-dataset-improvements-hn814c'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch {BRANCH} https://github.com/awg-yk/typhoon-wind-rainfall {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

## ② 必要なライブラリをインストール

In [ ]:
!pip install -q xarray netCDF4

## ③ ダウンロード実行（本体、日本域に切り出し済み）

**1時間程度かかります。**1件ずつ、さらに5日分ずつに分けて取得し、各行に経過時間と残り時間の目安が出ます。

504エラーが多発する場合は末尾に `--chunk-days 2` を追加（1回のリクエストをさらに軽くする）。地上気圧も欲しい場合は `--include-surface-pressure`、切り出す範囲を変えたい場合は `--north/--south/--west/--east` を追加してください。

最後まで走っても失敗が残った場合は、このセルをもう一度実行すると、失敗した月だけ再取得を試みます（成功済みのファイルはスキップされます）。

In [ ]:
LOCAL_DIR = '/content/jra3q_pressure'

!cd {REPO_DIR} && python scripts/download_jra3q_pressure.py --out-dir "{LOCAL_DIR}"

## ④ ZIPにまとめてPCへダウンロード

In [ ]:
import shutil
import pathlib

from google.colab import files as colab_files

files = list(pathlib.Path(LOCAL_DIR).glob('*.nc'))
total_mb = sum(f.stat().st_size for f in files) / 1e6
print(f'{len(files)} files, {total_mb:.1f} MB')

zip_base = '/content/jra3q_pressure_japan'
zip_path = shutil.make_archive(zip_base, 'zip', LOCAL_DIR)
print(f'{zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
colab_files.download(zip_path)

## ⑤ （最初に実行してください）無操作切断を遅らせる

③は1時間ほどかかるため、その間にColabが無操作と判断して切断することがあります。**③より先に**このセルを流しておくと、ブラウザのタブを開いたままにしている間は接続維持の合図を送り続けます（非公式の小技のため過信せず、切れたら①からやり直してください）。

In [ ]:
from IPython.display import Javascript, display

display(Javascript('''
function KeepAlive(){
  console.log("keep-alive ping");
  document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(KeepAlive, 60000);
'''))

## PCで受け取った後

`jra3q_pressure_japan.zip` を、リポジトリの `data/raw_jra3q/` フォルダなど好きな場所に展開してください。`data/raw_jra3q/` は `.gitignore` 済みなので、そのまま置いてもリポジトリには影響しません。